# 2단계: Q1 — 설정 순위가 데이터셋을 건너도 유지되는가

설정 10개 x 데이터셋 3개 x 시드 3개를 비교 단계 전처리(`Resize((256, 256))`, 자르지 않음)로 돌린다.

| 데이터셋 | 부류 | 학습 / 시험 |
|---|---|---|
| MVTec AD | 15 전부 | 3,629 / 1,725 |
| VisA | 12 전부, 공식 1cls 분할 | 8,659 / 2,162 |
| Real-IAD | 10 (시드 0으로 뽑음), C1, 단일 시점 분할 `realiad_jsons_sv` | 3,552 / 6,501 |

설정은 PatchCore 6개(ResNet18·WRN-50-2 x 층 (2,3)의 1%·10%, 층 (1,2,3)의 1%)와 PaDiM 4개다. 층 (1,2,3)의 10%는 계산 시간 때문에 뺐다(README 「Q1 설정」).

**시간.** 재현 CSV의 코어셋 시간을 큰 뱅크 쪽으로 늘려 잡은 추정으로 모두 약 35시간이다(MVTec 약 5, Real-IAD 약 8, VisA 약 22). 잰 값이 아니므로 3번 「짧은 실행」에서 가장 무거운 칸 하나를 먼저 잰다.

**순서.** 시드 0으로 세 데이터셋을 다 돌린 뒤 시드 1, 시드 2로 간다. 시드 0만 끝나도 순위 표를 한 번 볼 수 있다.

**끊겨도 된다.** 결과는 부류 하나가 끝날 때마다 Drive의 CSV에 적는다. 런타임이 끊기면 1·2번 셀과 3번의 첫 셀을 다시 돌린 뒤 4번 셀을 다시 돌리면 끝난 줄을 건너뛴다.

**실행 전 준비.** 런타임 GPU(L4), 보안 비밀 `GITHUB_TOKEN` 접근 허용. 01 노트북과 같다.

## 1. 코드와 패키지

In [1]:
import os, subprocess, sys

REPO_DIR = "/content/beyond-mvtec"
REPO_URL = "https://github.com/chlee1027/beyond-mvtec.git"
UPSTREAM_DIR = "/content/patchcore-inspection"
UPSTREAM_COMMIT = "fcaa92f124fb1ad74a7acf56726decd4b27cbcad"

from google.colab import userdata
token = userdata.get("GITHUB_TOKEN").strip()
auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "-q", auth_url, REPO_DIR], capture_output=True, text=True)
else:
    # 이미 받아 둔 레포는 GitHub 의 최신 main 으로 맞춘다(토큰 든 주소는 저장하지 않는다)
    r = subprocess.run(["git", "-C", REPO_DIR, "pull", "-q", "--ff-only", auth_url, "main"],
                       capture_output=True, text=True)
# 실패해도 오류 메시지에 토큰이 찍히지 않게 가린다
assert r.returncode == 0, "레포 받기 실패: " + r.stderr.replace(token, "***")
subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
if not os.path.isdir(UPSTREAM_DIR):
    subprocess.run(["git", "clone", "-q", "https://github.com/amazon-science/patchcore-inspection.git", UPSTREAM_DIR], check=True)
subprocess.run(["git", "-C", UPSTREAM_DIR, "checkout", "-q", UPSTREAM_COMMIT], check=True)

!pip uninstall -y -q sympy
!pip install -q sympy==1.13.3 faiss-gpu-cu12 timm click pretrainedmodels scikit-image scikit-learn scipy tqdm pandas pyyaml

sys.path[:0] = [os.path.join(REPO_DIR, "src"), os.path.join(UPSTREAM_DIR, "src")]
COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()

import faiss, torch, torchvision
assert torch.cuda.is_available(), "GPU 런타임이 아니다"
print(f"레포 {COMMIT} | Python {sys.version.split()[0]} | torch {torch.__version__} | torchvision {torchvision.__version__} | faiss {faiss.__version__}")
print(f"GPU {torch.cuda.get_device_name(0)} | VRAM {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert hasattr(faiss, "StandardGpuResources"), "faiss 의 GPU 판이 아니다"

from beyond_mvtec import data, experiment as ex, patchcore_adapter
config = ex.load_config(f"{REPO_DIR}/configs/experiments.yaml")
SETTINGS = ex.comparison_settings(config)
# 레포가 Q1 용 코드보다 옛 판이면 여기서 멈춘다(단일 시점 분할, 설정 10개, 순위 상관 함수, 코어셋 뒤 GPU 메모리 풀기)
assert len(SETTINGS) == 10 and hasattr(ex, "kendall_pairs") and config["datasets"]["real_iad"].get("split") == "sv" \
    and hasattr(patchcore_adapter, "_releasing_sampler"), \
    "옛 코드가 읽혔다. 런타임 > 세션 다시 시작 후 1번 셀부터 다시 돌린다"
print("설정", len(SETTINGS), "개:", [s["setting"] for s in SETTINGS])

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
레포 88fd4dc | Python 3.13.16 | torch 2.14.1+cu130 | torchvision 0.29.1+cu130 | faiss 1.14.1
GPU NVIDIA L4 | VRAM 23.7 GB
설정 10 개: ['patchcore_r18_L23_c1', 'patchcore_r18_L23_c10', 'patchcore_r18_L123_c1', 'patchcore_wrn50_L23_c1', 'patchcore_wrn50_L23_c10', 'patchcore_wrn50_L123_c1', 'padim_r18_L123_d100', 'padim_r18_L23_d100', 'padim_wrn50_L123_d550', 'padim_wrn50_L23_d550']


## 2. Drive와 데이터

In [2]:
import shutil, zipfile
from google.colab import drive
drive.mount("/content/drive")

MY = "/content/drive/MyDrive"
os.environ["TORCH_HOME"] = f"{MY}/torch_cache"
RESULT_DIR = f"{MY}/beyond_mvtec_results"
os.makedirs(RESULT_DIR, exist_ok=True)

def copy_once(src, dst):
    """Drive 에서 파일을 하나씩 읽으면 느려서 런타임 디스크로 한 번 옮긴다. 진행 표시는 없다."""
    if not os.path.isfile(f"{dst}/.copied"):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copytree(src, dst, dirs_exist_ok=True)
        open(f"{dst}/.copied", "w").close()
    print("Drive:", src, "-> 로컬:", dst)

MVTEC_DRIVE = next((p for p in [f"{MY}/anormaly_detection/mvtec", f"{MY}/mvtec", f"{MY}/MVTecAD"]
                    if os.path.isdir(f"{p}/bottle")), None)
VISA_DRIVE = next((p for p in [f"{MY}/visa", f"{MY}/VisA_20220922", f"{MY}/VisA"]
                   if os.path.isfile(f"{p}/split_csv/1cls.csv")), None)
assert MVTEC_DRIVE and VISA_DRIVE, (MVTEC_DRIVE, VISA_DRIVE)

ROOTS = {"mvtec_ad": "/content/data/mvtec_ad", "visa": "/content/data/visa", "real_iad": "/content/data/real_iad"}
copy_once(MVTEC_DRIVE, ROOTS["mvtec_ad"])
copy_once(VISA_DRIVE, ROOTS["visa"])

# Real-IAD: 분할 JSON 을 풀고, 부류 zip 에서 C1 사진과 마스크만 푼다
RI = config["datasets"]["real_iad"]
VIEW, RI_SPLIT = RI["views"][0], RI["split"]
os.makedirs(ROOTS["real_iad"], exist_ok=True)
if not os.path.isdir(f"{ROOTS['real_iad']}/realiad_jsons"):
    with zipfile.ZipFile(f"{MY}/realiad_jsons.zip") as z:
        z.extractall(f"{ROOTS['real_iad']}/realiad_jsons")
for category in RI["categories"]:
    data.prepare_real_iad(f"{MY}/Real_IAD/{category}.zip", ROOTS["real_iad"], category, view=VIEW)
print("Real-IAD", len(RI["categories"]), "부류:", RI["categories"], "| 시점", VIEW, "| 분할", RI_SPLIT)

CATEGORIES = {"mvtec_ad": data.MVTEC_CATEGORIES, "visa": data.visa_categories(ROOTS["visa"]),
              "real_iad": RI["categories"]}

Mounted at /content/drive
Drive: /content/drive/MyDrive/mvtec -> 로컬: /content/data/mvtec_ad
Drive: /content/drive/MyDrive/visa -> 로컬: /content/data/visa
Real-IAD 10 부류: ['bottle_cap', 'pcb', 'phone_battery', 'porcelain_doll', 'regulator', 'sim_card_set', 'switch', 'usb', 'wooden_beads', 'woodstick'] | 시점 C1 | 분할 sv


In [3]:
import pandas as pd

rows = []
for dataset, categories in CATEGORIES.items():
    for category in categories:
        split = ex.list_split(dataset, ROOTS[dataset], category, VIEW, real_iad_split=RI_SPLIT)
        rows.append({"dataset": dataset, "category": category, "train": len(split.train),
                     "test": len(split.test), "test_anomaly": sum(s.label for s in split.test)})
counts = pd.DataFrame(rows)
display(counts.groupby("dataset")[["train", "test", "test_anomaly"]].sum())

# 00 노트북에서 센 장수와 같아야 한다
total = counts.groupby("dataset")[["train", "test"]].sum()
assert tuple(total.loc["mvtec_ad"]) == (3629, 1725), tuple(total.loc["mvtec_ad"])
assert tuple(total.loc["visa"]) == (8659, 2162), tuple(total.loc["visa"])
expected_ri = {"bottle_cap": (370, 633), "pcb": (279, 723), "phone_battery": (350, 650), "porcelain_doll": (402, 598),
               "regulator": (477, 543), "sim_card_set": (305, 701), "switch": (267, 743), "usb": (354, 656),
               "wooden_beads": (305, 696), "woodstick": (443, 558)}
got_ri = {r.category: (r.train, r.test) for r in counts[counts.dataset == "real_iad"].itertuples()}
assert got_ri == expected_ri, got_ri
print("장수가 00 노트북과 같다")
display(counts[counts.dataset == "real_iad"])

,train,test,test_anomaly
dataset,,,
mvtec_ad,3629,1725,1258
real_iad,3552,6501,2956
visa,8659,2162,1200


장수가 00 노트북과 같다


,dataset,category,train,test,test_anomaly
27,real_iad,bottle_cap,370,633,264
28,real_iad,pcb,279,723,445
29,real_iad,phone_battery,350,650,301
30,real_iad,porcelain_doll,402,598,196
31,real_iad,regulator,477,543,66
32,real_iad,sim_card_set,305,701,396
33,real_iad,switch,267,743,477
34,real_iad,usb,354,656,303
35,real_iad,wooden_beads,305,696,392
36,real_iad,woodstick,443,558,116


## 3. 짧은 실행: 가장 무거운 칸 먼저

학습이 가장 많은 VisA `pcb3`(905장), 시드 0에서 두 설정을 먼저 돈다. 결과는 4번과 같은 CSV에 적히므로 4번에서 다시 돌지 않는다.

- `patchcore_wrn50_L123_c1`: 코어셋 전 뱅크가 905 x 4,096 = 370만 조각 x 1,024차원, float32로 15.2 GB이고 코어셋을 고르기 전 128차원으로 사영할 때 GPU에 올라간다(원본 코드를 읽고 센 것). 코어셋 시간 추정은 약 760초다.
  첫 실행(2026-11-08)에서 코어셋 뒤 FAISS 의 GPU 임시 메모리 요청(1.5 GiB)이 실패했다. 원본 샘플러가 사영을 기울기 계산을 끄지 않고 해서 이 뱅크가 자동 미분 그래프에 묶여 남았고, 풀린 뒤에도 PyTorch 캐시에 있었다. 지금은 사영을 기울기 없이 하고 코어셋이 끝나면 캐시를 비운다(고르는 값은 같다).
- `padim_wrn50_L123_d550`: 재현(224)에서 최대 14.3 GB였다. 256에서는 위치 수가 3,136에서 4,096으로 늘어 약 18.7 GB로 추정한다.

**넘어가는 기준.** 둘 다 오류 없이 끝나고, PatchCore의 `fit_sec`가 600~1,400초 안이면 4번으로 바로 간다. 메모리 오류가 나거나 `fit_sec`가 이 범위를 벗어나면 4번을 돌리지 말고 이 셀 출력을 보낸다.

In [4]:
import resource

PREPROCESS = config["preprocess"]["compare"]
OPTIONS = dict(device="cuda", batch_size=8, num_workers=4, faiss_on_gpu=True, commit=COMMIT,
               view=VIEW, real_iad_split=RI_SPLIT, save_dir=f"{RESULT_DIR}/q1_scores")
CSV = {d: f"{RESULT_DIR}/q1_{d}.csv" for d in ROOTS}

In [5]:
heavy = [s for s in SETTINGS if s["setting"] in ("patchcore_wrn50_L123_c1", "padim_wrn50_L123_d550")]
ex.run_grid(heavy, "visa", ROOTS["visa"], ["pcb3"], [0], "compare", PREPROCESS, CSV["visa"], **OPTIONS)

done = pd.read_csv(CSV["visa"])
display(done[(done.category == "pcb3") & (done.seed == 0) & done.setting.isin([s["setting"] for s in heavy])]
        [["setting", "image_auroc", "pixel_auroc", "pro", "bank_size", "fit_sec", "predict_sec", "metric_sec", "peak_gpu_gb"]])
print(f"호스트 메모리 최대 {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

,setting,image_auroc,pixel_auroc,pro,bank_size,fit_sec,predict_sec,metric_sec,peak_gpu_gb
0,patchcore_wrn50_L123_c1,0.992178,0.992299,0.935518,37068.0,843.06,35.53,6.93,17.38
1,padim_wrn50_L123_d550,0.820693,0.988009,0.845788,NaN,44.84,2.99,7.61,17.70


호스트 메모리 최대 1.1 GB


## 4. Q1 실행

시드 하나마다 MVTec -> Real-IAD -> VisA 순서로 돈다. 점수와 라벨은 `q1_scores/`에 남기고 이상 지도는 남기지 않는다(Real-IAD 한 번이 float16으로 약 0.85 GB).

In [6]:
for seed in config["seeds"]:
    for dataset in ["mvtec_ad", "real_iad", "visa"]:
        print(f"===== 시드 {seed} | {dataset} =====")
        ex.run_grid(SETTINGS, dataset, ROOTS[dataset], CATEGORIES[dataset], [seed], "compare", PREPROCESS,
                    CSV[dataset], **OPTIONS)

===== 시드 0 | mvtec_ad =====
===== 시드 0 | real_iad =====
===== 시드 0 | visa =====
===== 시드 1 | mvtec_ad =====
===== 시드 1 | real_iad =====
===== 시드 1 | visa =====
===== 시드 2 | mvtec_ad =====
===== 시드 2 | real_iad =====
===== 시드 2 | visa =====


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1`. You can also use `weights=Wide_ResNet50_2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 92569/92569 [07:18<00:00, 211.20it/s]


[59/120] patchcore_wrn50_L23_c10 pcb4 s2 I=0.9952 P=0.9855 PRO=0.8898 fit=477.89s pred=13.81s gpu=4.58GB


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1`. You can also use `weights=Wide_ResNet50_2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 46080/46080 [01:49<00:00, 420.90it/s]


[60/120] patchcore_wrn50_L23_c10 pipe_fryum s2 I=0.9990 P=0.9924 PRO=0.9534 fit=133.2s pred=7.77s gpu=2.42GB


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1`. You can also use `weights=Wide_ResNet50_2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 36864/36864 [11:27<00:00, 53.61it/s]


... (19 outputs left out)

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1`. You can also use `weights=Wide_ResNet50_2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 37027/37027 [11:32<00:00, 53.50it/s]


[71/120] patchcore_wrn50_L123_c1 pcb4 s2 I=0.9982 P=0.9794 PRO=0.8650 fit=868.25s pred=34.42s gpu=17.41GB


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1`. You can also use `weights=Wide_ResNet50_2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 18432/18432 [02:51<00:00, 107.69it/s]


[72/120] patchcore_wrn50_L123_c1 pipe_fryum s2 I=0.9996 P=0.9918 PRO=0.9678 fit=258.36s pred=22.91s gpu=8.79GB
[73/120] padim_r18_L123_d100 candle s2 I=0.8914 P=0.9812 PRO=0.9406 fit=10.53s pred=2.26s gpu=0.65GB
[74/120] padim_r18_L123_d100 capsules s2 I=0.6250 P=0.9139 PRO=0.5522 fit=3.88s pred=2.08s gpu=0.66GB
[75/120] padim_r18_L123_d100 cashew s2 I=0.9074 P=0.9823 PRO=0.8708 fit=3.61s pred=2.11s gpu=0.65GB
[76/120] padim_r18_L123_d100 chewinggum s2 I=0.9866 P=0.9885 PRO=0.8345 fit=3.3s pred=1.94s gpu=0.66GB
[77/120] padim_r18_L123_d100 fryum s2 I=0.8912 P=0.9673 PRO=0.8088 fit=3.24s pred=2.0s gpu=0.65GB
[78/120] padim_r18_L123_d100 macaroni1 s2 I=0.7983 P=0.9819 PRO=0.9031 fit=5.28s pred=2.25s gpu=0.65GB
[79/120] padim_r18_L123_d100 macaroni2 s2 I=0.6778 P=0.9549 PRO=0.7411 fit=5.33s pred=2.16s gpu=0.65GB
[80/120] padim_r18_L123_d100 pcb1 s2 I=0.8890 P=0.9914 PRO=0.8879 fit=5.82s pred=2.21s gpu=0.67GB
[81/120] padim_r18_L123_d100 pcb2 s2 I=0.7995 P=0.9824 PRO=0.8525 fit=5.6s pred=2

## 5. 순위와 순위 상관

이미지 AUROC로 데이터셋마다 설정 10개의 순위를 매기고, 데이터셋 쌍마다 Kendall tau-b와 뒤집힌 쌍 수(45쌍 중)를 낸다. 시드 하나씩 따로 낸 값도 같이 본다. 시드 0만 끝난 상태에서도 돌릴 수 있다.

In [7]:
paths = [p for p in CSV.values() if os.path.isfile(p)]
summary = ex.summarize(paths)
display(summary[["dataset", "setting", "image_auroc_mean", "image_auroc_std", "pixel_auroc_mean", "pro_mean",
                 "n_seeds", "n_categories"]].round(2))
expected_n = {d: len(c) for d, c in CATEGORIES.items()}
complete = all(((summary.dataset == d) & (summary.n_seeds == 3) & (summary.n_categories == n)).sum() == len(SETTINGS)
               for d, n in expected_n.items())
print("칸이 다 찼나:", complete)

values, ranks = ex.rank_by_dataset(paths)
print("이미지 AUROC (%, 시드 평균)")
display(values.round(2))
print("순위 (1 = 가장 높음)")
display(ranks)
print("Kendall tau-b, 시드 평균 값으로")
display(ex.kendall_pairs(values).round(3))
print("Kendall tau-b, 시드 하나씩")
display(ex.kendall_per_seed(paths).round(3))

,dataset,setting,image_auroc_mean,image_auroc_std,pixel_auroc_mean,pro_mean,n_seeds,n_categories
0,mvtec_ad,padim_r18_L123_d100,90.96,0.42,96.79,91.88,3,15
1,mvtec_ad,padim_r18_L23_d100,91.56,1.39,96.27,90.63,3,15
2,mvtec_ad,padim_wrn50_L123_d550,95.15,0.99,97.76,93.88,3,15
3,mvtec_ad,padim_wrn50_L23_d550,95.47,0.22,97.72,93.70,3,15
4,mvtec_ad,patchcore_r18_L123_c1,95.01,0.25,96.57,92.56,3,15
5,mvtec_ad,patchcore_r18_L23_c1,97.01,0.07,96.80,91.61,3,15
6,mvtec_ad,patchcore_r18_L23_c10,96.87,0.08,97.40,92.94,3,15
7,mvtec_ad,patchcore_wrn50_L123_c1,98.20,0.05,98.06,94.53,3,15
8,mvtec_ad,patchcore_wrn50_L23_c1,98.37,0.13,98.13,93.60,3,15
9,mvtec_ad,patchcore_wrn50_L23_c10,98.55,0.04,98.38,94.43,3,15


칸이 다 찼나: True
이미지 AUROC (%, 시드 평균)


dataset,mvtec_ad,real_iad,visa
setting,,,
padim_r18_L123_d100,90.96,81.62,83.60
padim_r18_L23_d100,91.56,83.41,84.46
padim_wrn50_L123_d550,95.15,88.03,87.96
padim_wrn50_L23_d550,95.47,88.67,87.77
patchcore_r18_L123_c1,95.01,92.11,95.85
patchcore_r18_L23_c1,97.01,89.03,93.23
patchcore_r18_L23_c10,96.87,90.39,93.10
patchcore_wrn50_L123_c1,98.20,93.55,96.60
patchcore_wrn50_L23_c1,98.37,91.83,93.64


순위 (1 = 가장 높음)


dataset,mvtec_ad,real_iad,visa
setting,,,
padim_r18_L123_d100,10.0,10.0,10.0
padim_r18_L23_d100,9.0,9.0,9.0
padim_wrn50_L123_d550,7.0,8.0,7.0
padim_wrn50_L23_d550,6.0,7.0,8.0
patchcore_r18_L123_c1,8.0,3.0,2.0
patchcore_r18_L23_c1,4.0,6.0,4.0
patchcore_r18_L23_c10,5.0,5.0,6.0
patchcore_wrn50_L123_c1,3.0,1.0,1.0
patchcore_wrn50_L23_c1,2.0,4.0,3.0


Kendall tau-b, 시드 평균 값으로


,pair,n_settings,n_pairs,concordant,discordant,tied,tau_b,p_value
0,mvtec_ad - real_iad,10,45,37,8,0,0.644,0.009
1,mvtec_ad - visa,10,45,34,11,0,0.511,0.047
2,real_iad - visa,10,45,40,5,0,0.778,0.001


Kendall tau-b, 시드 하나씩


,pair,n_settings,n_pairs,concordant,discordant,tied,tau_b,p_value,seed
0,mvtec_ad - real_iad,10,45,38,7,0,0.689,0.005,0
1,mvtec_ad - visa,10,45,36,9,0,0.600,0.017,0
2,real_iad - visa,10,45,41,4,0,0.822,0.000,0
3,mvtec_ad - real_iad,10,45,36,9,0,0.600,0.017,1
4,mvtec_ad - visa,10,45,34,11,0,0.511,0.047,1
5,real_iad - visa,10,45,39,6,0,0.733,0.002,1
6,mvtec_ad - real_iad,10,45,36,9,0,0.600,0.017,2
7,mvtec_ad - visa,10,45,37,8,0,0.644,0.009,2
8,real_iad - visa,10,45,42,3,0,0.867,0.000,2


In [8]:
# 시간과 메모리: 설정 x 데이터셋, 부류 한 번 평균
df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
display(df.groupby(["dataset", "setting"])[["fit_sec", "predict_sec", "metric_sec", "peak_gpu_gb"]].mean().round(1))
print("더한 시간(시간):", round((df.fit_sec + df.predict_sec + df.metric_sec).sum() / 3600, 1))

fit_sec  predict_sec  metric_sec  \
dataset  setting                                                     
mvtec_ad padim_r18_L123_d100          3.8          2.4         4.2   
         padim_r18_L23_d100           3.6          2.4         4.2   
         padim_wrn50_L123_d550       16.1          2.6         4.2   
         padim_wrn50_L23_d550         5.4          2.5         4.2   
         patchcore_r18_L123_c1       99.4         16.4         4.2   
         patchcore_r18_L23_c1        16.2          5.2         4.1   
         patchcore_r18_L23_c10       48.3          5.8         4.1   
         patchcore_wrn50_L123_c1    102.0         17.8         4.1   
         patchcore_wrn50_L23_c1      17.3          5.7         4.1   
         patchcore_wrn50_L23_c10     49.5          6.3         4.1   
real_iad padim_r18_L123_d100          2.8          4.2        29.2   
         padim_r18_L23_d100           2.5          4.2        29.0   
         padim_wrn50_L123_d550       19.7          6.6        28.9   
         padim_wrn50_L23_d550         5.9          4.8        28.7   
         patchcore_r18_L123_c1      179.1         84.9        28.9   
         patchcore_r18_L23_c1        24.9         22.5        29.9   
         patchcore_r18_L23_c10       90.1         27.1        29.6   
         patchcore_wrn50_L123_c1    180.7         93.7        29.6   
         patchcore_wrn50_L23_c1      26.8         25.8        30.4   
         patchcore_wrn50_L23_c10     91.6         30.6        29.2   
visa     padim_r18_L123_d100          5.0          2.1         6.9   
         padim_r18_L23_d100           4.4          2.0         6.9   
         padim_wrn50_L123_d550       32.3          2.7         6.9   
         padim_wrn50_L23_d550         9.6          2.2         6.8   
         patchcore_r18_L123_c1      616.6         28.8         7.0   
         patchcore_r18_L23_c1        65.5          6.9         6.9   
         patchcore_r18_L23_c10      340.5         10.6         6.8   
         patchcore_wrn50_L123_c1    621.9         29.9         6.9   
         patchcore_wrn50_L23_c1      69.6          8.3         7.1   
         patchcore_wrn50_L23_c10    344.4         12.1         7.0   

                                  peak_gpu_gb  
dataset  setting                               
mvtec_ad padim_r18_L123_d100              0.7  
         padim_r18_L23_d100               0.3  
         padim_wrn50_L123_d550           17.8  
         padim_wrn50_L23_d550             6.5  
         patchcore_r18_L123_c1            4.7  
         patchcore_r18_L23_c1             1.2  
         patchcore_r18_L23_c10            1.2  
         patchcore_wrn50_L123_c1          5.2  
         patchcore_wrn50_L23_c1           1.5  
         patchcore_wrn50_L23_c10          1.5  
real_iad padim_r18_L123_d100              0.7  
         padim_r18_L23_d100               0.3  
         padim_wrn50_L123_d550           17.7  
         padim_wrn50_L23_d550             6.5  
         patchcore_r18_L123_c1            6.8  
         patchcore_r18_L23_c1             1.7  
         patchcore_r18_L23_c10            1.7  
         patchcore_wrn50_L123_c1          7.0  
         patchcore_wrn50_L23_c1           2.0  
         patchcore_wrn50_L23_c10          2.0  
visa     padim_r18_L123_d100              0.7  
         padim_r18_L23_d100               0.3  
         padim_wrn50_L123_d550           17.8  
         padim_wrn50_L23_d550             6.5  
         patchcore_r18_L123_c1           13.7  
         patchcore_r18_L23_c1             3.5  
         patchcore_r18_L23_c10            3.5  
         patchcore_wrn50_L123_c1         13.9  
         patchcore_wrn50_L23_c1           3.7  
         patchcore_wrn50_L23_c10          3.7

더한 시간(시간): 38.9


## 6. 결과를 레포로

Drive `beyond_mvtec_results/`의 `q1_mvtec_ad.csv`, `q1_real_iad.csv`, `q1_visa.csv`를 레포 `results/`에 넣는다. `q1_scores/`는 올리지 않는다.